# 05 · Export for HLS (one checkpoint → every artifact)
Produces, from a single checkpoint and a **real** test window:
* `verification_data/*.txt` golden intermediates (`conv1d_out`, `l0_rope_out`, `l0_add_norm1_out`, …, one set per encoder layer, `pool_out`, `full_output`) and float weights `full_*.txt`
* `weights/*.h` quantised headers: `conv1d_weights.h`, `encoder_weights.h` (all layers stacked), `fc_weights.h`, plus `results.h` (float and fixed-point references) and `input_output.h`
* `model_config.h` (dimensions, number of layers, fixed-point formats) and `rope_tables.h`
* `hil_verification_samples/` real windows + float outputs + numpy fixed-point outputs + manifest
* `normalization_constants.{json,h}`: per-channel EMG scale, glove min/max, mu, causal SOS filter, `DELTA_SCALE`, I/O sizes
* `fixed_point_expectation.json`: the per-layer error a *correct* HLS build should reproduce

Install with `python hls_code/scripts/install_export.py <export dir>`, then `cd hls_code && make run`. The export refuses models the
HLS cannot implement (`ec.hls_unsupported(cfg)` lists why). For the 200 ms deep models use notebook 09, which also does QAT.

In [ ]:
# --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
import os, sys, glob, json
for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
    if os.path.exists(os.path.join(p, 'emg_common.py')):
        sys.path.insert(0, os.path.abspath(p)); break
else:
    raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
# processed data: output of 01_preprocess_split (same session) or attached as a dataset
candidates = [f'{WORK}/db2_processed'] + sorted(glob.glob('/kaggle/input/*/db2_processed')) + sorted(glob.glob('/kaggle/input/*'))
npz_files = []
for c in candidates:
    npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
    if npz_files: break
assert npz_files, 'no processed data found: run 01_preprocess_split first or attach its output'
print(len(npz_files), 'subjects:', [os.path.basename(os.path.dirname(f)) for f in npz_files])

In [ ]:
ckpts = sorted(glob.glob(f'{WORK}/outputs/*.pt') + glob.glob('/kaggle/input/*/outputs/*.pt') + glob.glob('/kaggle/input/*/*.pt'))
CKPT = os.environ.get('EMG_CKPT', ckpts[-1] if ckpts else None); assert CKPT, 'no checkpoint found'
model, cfg, extra = ec.load_checkpoint(CKPT); model.eval()
cfg.act_fmt, cfg.weight_fmt, cfg.rope_fmt = (10, 3), (10, 3), (8, 2)      # formats to quantise headers/tables with
cfg.export_weight_type = 'data_t'                                         # 'weight_t' once types.h has the typedef
EXPORT_DIR = f'{WORK}/hls_export/{os.path.splitext(os.path.basename(CKPT))[0]}'
print(CKPT, '| variant', cfg.variant, '| heads', cfg.heads, '| cond_prev', cfg.cond_prev, '| tokens', cfg.num_tokens)

In [ ]:
# ---- a real test window for the golden dumps + N real HIL windows spread over the test split ----
d = np.load(npz_files[0]); test_idx = np.where((d['split'] == 1) & (d['has_prev'] == 1))[0]
i = int(test_idx[len(test_idx) // 3])
sample_x = d['sEMG'][i].T.astype(np.float64)                       # [C, seq_len] channel-major = HLS input layout
sample_prev = d['angles_prev'][i].astype(np.float64) if cfg.cond_prev else None
N_HIL = int(os.environ.get('EMG_N_HIL', 40))
hil = [(d['sEMG'][j].T.astype(np.float64), d['angles_prev'][j].astype(np.float64) if cfg.cond_prev else None, int(d['restim'][j]), int(d['rep'][j]))
       for j in test_idx[np.linspace(0, len(test_idx) - 1, N_HIL).astype(int)]]
report = ec.export_hls(model, cfg, EXPORT_DIR, sample_x, sample_prev, hil, ec.load_norm_stats(npz_files[0]))
display(pd.DataFrame(report).T)
print(open(f'{EXPORT_DIR}/model_config.h').read())

In [ ]:
# ---- package for download and print the install instructions ----
import shutil; zip_path = shutil.make_archive(EXPORT_DIR, 'zip', EXPORT_DIR); print('archive:', zip_path)
print(f"""
Install locally (repository root):
  unzip {os.path.basename(zip_path)} -d hls_export
  python hls_code/scripts/install_export.py hls_export
  cd hls_code && make clean && make run     # every layer must PASS against the fixed-point reference
install_export.py replaces model_config.h, rope_tables.h, weights/, verification_data/, hil_verification_samples/
and the firmware's normalization_constants.h, so nothing from the previous model is left behind.""")